In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# target components
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat04','Tomcat03',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# helper to read and preprocess a file
def load_and_prep(fname, kpi_col):
    df = pd.read_csv(fname)
    # convert timestamp to timezone-aware datetime in Asia/Shanghai (UTC+8)
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    # keep only wanted components
    df = df[df['cmdb_id'].isin(components)].copy()
    # ensure numeric values
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    # drop rows with missing value or missing kpi
    df = df.dropna(subset=['value', kpi_col])
    return df

# aggregation function
def aggregate_series(df, kpi_col, source_name):
    g = df.groupby(['cmdb_id', kpi_col])['value']
    agg_basic = g.agg(count='count', min='min', max='max', mean='mean')
    # quantiles
    q = g.quantile([0.5, 0.9, 0.95, 0.99]).unstack(level=-1)
    q = q.rename(columns={0.5:'p50', 0.9:'p90', 0.95:'p95', 0.99:'p99'})
    agg = agg_basic.join(q)
    agg = agg.reset_index().rename(columns={kpi_col: 'kpi'})
    agg['source'] = source_name
    # reorder columns
    agg = agg[['source','cmdb_id','kpi','count','min','max','mean','p50','p90','p95','p99']]
    # round numeric columns for compactness
    num_cols = ['min','max','mean','p50','p90','p95','p99']
    agg[num_cols] = agg[num_cols].round(6)
    return agg

# load files
metric_df = load_and_prep('metric.csv', 'kpi_name')
trace_df  = load_and_prep('trace.csv', 'trace_name')
log_df    = load_and_prep('log.csv', 'log_name')

# aggregate each
metric_agg_full = aggregate_series(metric_df, 'kpi_name', 'metric')
trace_agg_full  = aggregate_series(trace_df,  'trace_name', 'trace')
log_agg_full    = aggregate_series(log_df,    'log_name',   'log')

# combined full aggregated results
combined_agg_full = pd.concat([metric_agg_full, trace_agg_full, log_agg_full], ignore_index=True)

# sort and limit to top 50 rows (sorted by source, cmdb_id, kpi)
combined_agg_full = combined_agg_full.sort_values(['source','cmdb_id','kpi'], ascending=[True,True,True]).reset_index(drop=True)
combined_head = combined_agg_full.head(50)

# display the compact combined table (top 50)
combined_head

```
Out[1]:
```
```python
# Summarize the aggregated results in plain English as a single compact string.
# Reuse existing variables: combined_agg_full, metric_agg_full, trace_agg_full, log_agg_full

# counts of aggregated (cmdb_id, kpi) series per source
src_counts = combined_agg_full['source'].value_counts().to_dict()

# components present per source
components_by_src = combined_agg_full.groupby('source')['cmdb_id'].unique().apply(list).to_dict()

# top 3 series by mean value for each source (compact info)
def top_by_mean(df, source, n=3):
    sub = df[df['source']==source].copy()
    if sub.empty:
        return []
    top = sub.sort_values('mean', ascending=False).head(n)[['cmdb_id','kpi','count','mean']].round(6)
    return [f"{row.cmdb_id} | {row.kpi} | count={int(row.count)} | mean={row.mean}" for row in top.itertuples(index=False)]

top_examples = {s: top_by_mean(combined_agg_full, s, 3) for s in ['metric','trace','log']}

summary = (
    f"Aggregation summary:\n"
    f"- Total aggregated series: {len(combined_agg_full)} across sources.\n"
    f"- Per-source series counts: " + ", ".join([f"{s}: {src_counts.get(s,0)}" for s in ['metric','trace','log']]) + ".\n"
    f"- Components present by source:\n"
    + "\n".join([f"  * {s}: {', '.join(sorted(components_by_src.get(s,[])) ) or 'none'}" for s in ['metric','trace','log']]) + "\n\n"
    f"Top example series by mean value per source (cmdb_id | kpi | count | mean):\n"
    + "\n".join([f"  {s}:\n    " + ("\n    ".join(top_examples[s]) if top_examples[s] else "none") for s in ['metric','trace','log']])
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source   cmdb_id                                                kpi  count        min        max         mean        p50         p90         p95          p99
0      log   Mysql01                                    log.error_count     31     0.0000     0.0000     0.000000     0.0000     0.00000     0.00000     0.000000
1      log   Mysql01                                      log.row_count     31     2.0000    43.0000    14.806452    13.0000    27.00000    35.00000    41.500000
2      log  Tomcat01                                    log.error_count     31     0.0000     0.0000     0.000000     0.0000     0.00000     0.00000     0.000000
3      log  Tomcat01                                   log.gc_pause_max     24     0.0400     0.1900     0.083750     0.0800     0.10000     0.10850     0.171600
4      log  Tomcat01                                      log.row_count     31     7.0000   557.0000   480.096774   504.0000   550.00000   553.00000   556.100000
..     ...       ...                                                ...    ...        ...        ...          ...        ...         ...         ...          ...
45  metric      IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...      9     0.0042     0.0042     0.004200     0.0042     0.00420     0.00420     0.004200
46  metric      IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedS...      8    15.0000    15.0000    15.000000    15.0000    15.00000    15.00000    15.000000
47  metric      IG01  OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace      8  9180.0000  9180.0000  9180.000000  9180.0000  9180.00000  9180.00000  9180.000000
48  metric      IG01        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity      9     3.2337     3.2338     3.233711     3.2337     3.23372     3.23376     3.233792
49  metric      IG01  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...      9     0.2202     0.2202     0.220200     0.2202     0.22020     0.22020     0.220200

[50 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables from kernel: combined_agg_full, metric_df, trace_df, log_df
# Ensure these exist
_combined = combined_agg_full

# Time window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

# Prepare thresholds dataframe from combined_agg_full (use p95 and p50 as provided)
thresholds = _combined[['source','cmdb_id','kpi','p95','p50']].copy()

# We need p5. If not present in combined_agg_full, compute global p5 per series using the full raw series (allowed as global pre-filtering).
# Build full raw combined dataframe with unified column names
def unify_raw(df, source, kpi_col):
    d = df.copy()
    d = d[['timestamp','dt','cmdb_id','value', kpi_col]].rename(columns={kpi_col: 'kpi'})
    d['source'] = source
    return d

raw_metric = unify_raw(metric_df, 'metric', 'kpi_name')
raw_trace  = unify_raw(trace_df,  'trace',  'trace_name')
raw_log    = unify_raw(log_df,    'log',    'log_name')

raw_all = pd.concat([raw_metric, raw_trace, raw_log], ignore_index=True)

# Compute global p5 (5th percentile) for each (source, cmdb_id, kpi) using full series
# Limit to series that exist in thresholds to avoid extra work
key_series = thresholds[['source','cmdb_id','kpi']].drop_duplicates()
# Merge to raw_all to filter only relevant series
merged = raw_all.merge(key_series, on=['source','cmdb_id','kpi'], how='inner')

# compute p5 per group
p5_series = merged.groupby(['source','cmdb_id','kpi'])['value'].quantile(0.05).reset_index().rename(columns={'value':'p5'})

# Merge p5 into thresholds
thresholds = thresholds.merge(p5_series, on=['source','cmdb_id','kpi'], how='left')

# Some p5 might be NaN if no raw rows existed in raw files for that series; keep as NaN (we'll skip low detection for those)
# Now filter raw rows within the requested time window and only those series present in thresholds
# Ensure dt column exists and is timezone-aware Asia/Shanghai
for df in (raw_all,):
    if df['dt'].dtype == object:
        # attempt conversion if needed (should already be set)
        df['dt'] = pd.to_datetime(df['dt']).dt.tz_localize('UTC').dt.tz_convert(tz)

windowed = raw_all[(raw_all['dt'] >= start_dt) & (raw_all['dt'] <= end_dt)].copy()

# Keep only series present in thresholds
windowed = windowed.merge(thresholds[['source','cmdb_id','kpi','p95','p5']], on=['source','cmdb_id','kpi'], how='inner')

# Label anomalies:
# - high if value >= p95
# - low if value <= p5 (only if p5 is not null)
def label_row(r):
    high = (pd.notna(r['p95'])) and (r['value'] >= r['p95'])
    low  = (pd.notna(r['p5']))  and (r['value'] <= r['p5'])
    if high and low:
        return 'both'
    if high:
        return 'high'
    if low:
        return 'low'
    return None

windowed['anomaly_type'] = windowed.apply(label_row, axis=1)
anomalies = windowed[windowed['anomaly_type'].notna()].copy()

# For severity calculations, we need p95 and p5 per row (already merged). Compute severity metrics per row:
# For high: ratio = value / p95 (guard divide by zero)
anomalies['severity_high'] = np.where(anomalies['anomaly_type'].isin(['high','both']),
                                      anomalies['value'] / anomalies['p95'].replace({0: np.nan}),
                                      np.nan)
# For low: ratio = p5 / value (guard divide by zero)
anomalies['severity_low'] = np.where(anomalies['anomaly_type'].isin(['low','both']),
                                     anomalies['p5'] / anomalies['value'].replace({0: np.nan}),
                                     np.nan)

# Aggregate anomalies per series
agg = anomalies.groupby(['source','cmdb_id','kpi']).agg(
    anomaly_count = ('anomaly_type','count'),
    earliest_anomaly_timestamp = ('dt','min'),
    anomaly_types = ('anomaly_type', lambda s: sorted(set(s))),
    max_value = ('value','max'),
    min_value = ('value','min'),
    max_severity_high = ('severity_high', lambda s: np.nanmax(s.dropna()) if s.notna().any() else np.nan),
    max_severity_low  = ('severity_low',  lambda s: np.nanmax(s.dropna()) if s.notna().any() else np.nan)
).reset_index()

# Derive final anomaly_type field ('high', 'low', or 'both')
def derive_type(types_list):
    if len(types_list)==0:
        return None
    tset = set(types_list)
    if tset == {'high'}:
        return 'high'
    if tset == {'low'}:
        return 'low'
    return 'both'

agg['anomaly_type'] = agg['anomaly_types'].apply(derive_type)
agg = agg.drop(columns=['anomaly_types'])

# severity: max of high and low severities (ignore NaN)
agg['severity'] = agg[['max_severity_high','max_severity_low']].max(axis=1)

# Add p95 and p5 from thresholds for reference
agg = agg.merge(thresholds[['source','cmdb_id','kpi','p95','p5']], on=['source','cmdb_id','kpi'], how='left')

# Format earliest_anomaly_timestamp as ISO string in UTC+8 for compact display
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Round numeric columns for compactness
num_cols = ['max_value','min_value','severity','p95','p5','max_severity_high','max_severity_low']
for c in num_cols:
    if c in agg.columns:
        agg[c] = agg[c].round(6)

# Prepare final anomalies_full (full aggregated anomalies table) and top20 by anomaly_count desc
anomalies_full = agg.sort_values(['anomaly_count','severity'], ascending=[False,False]).reset_index(drop=True)

top20 = anomalies_full.head(20)[['source','cmdb_id','kpi','anomaly_count','earliest_anomaly_timestamp','anomaly_type','max_value','min_value','severity','p95','p5']]

# Display compact top20 result
top20, anomalies_full.shape

```
Out[2]:
```
```python
# Summarize anomaly detection results in plain English using existing kernel variable anomalies_full
# The summary will be returned as a single compact string.

tz_note = "UTC+8"
time_window = "2021-03-24 14:30 to 2021-03-24 15:00 (UTC+8)"

total_series = int(anomalies_full.shape[0])
earliest_anomaly = anomalies_full['earliest_anomaly_timestamp'].min()
latest_anomaly = anomalies_full['earliest_anomaly_timestamp'].max()

# Top affected components (by number of anomalous series)
top_components = anomalies_full['cmdb_id'].value_counts().head(5)
top_components_str = "; ".join([f"{cmdb}={int(cnt)}" for cmdb,cnt in top_components.items()])

# Anomaly type distribution
atype_counts = anomalies_full['anomaly_type'].value_counts().to_dict()
atype_str = ", ".join([f"{k}={v}" for k,v in atype_counts.items()])

# Severity statistics (compact)
severity_series = anomalies_full['severity'].dropna()
severity_max = float(severity_series.max()) if not severity_series.empty else None
severity_median = float(severity_series.median()) if not severity_series.empty else None

# Example top 5 series by anomaly_count (already sorted in anomalies_full)
examples = anomalies_full.head(5)[['source','cmdb_id','kpi','anomaly_count','anomaly_type','severity']].copy()
examples['severity'] = examples['severity'].round(6)
examples_list = [f"{r.source}|{r.cmdb_id}|{r.kpi}|count={int(r.anomaly_count)}|type={r.anomaly_type}|sev={r.severity}"
                 for r in examples.itertuples(index=False)]

summary = (
    f"Anomaly detection summary for window {time_window}:\n"
    f"- Total anomalous series detected: {total_series}.\n"
    f"- Time range of reported anomalies: earliest={earliest_anomaly}, latest={latest_anomaly} ({tz_note}).\n"
    f"- Top affected components (by anomalous series): {top_components_str}.\n"
    f"- Anomaly types distribution: {atype_str} (values: high / low / both).\n"
    f"- Severity (ratio) median={severity_median}, max={severity_max} (rounded).\n"
    f"- Top example anomalous series (source|cmdb_id|kpi|count|type|severity):\n  - " + "\n  - ".join(examples_list) + "\n\n"
    f"Notes: thresholds p95 and p5 from the precomputed aggregated table were used. 'both' means that within the window the series had values meeting both high and low criteria."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                                kpi  anomaly_count earliest_anomaly_timestamp anomaly_type     max_value     min_value  severity           p95            p5
0   metric      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31    2021-03-24 14:30:00 CST         both  5.886977e+07  5.886734e+07  1.000021  5.886855e+07  5.886734e+07
1   metric      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31    2021-03-24 14:30:00 CST         both  2.040136e+09  2.040136e+09  1.000000  2.040136e+09  2.040136e+09
2   metric      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31    2021-03-24 14:30:00 CST         both  5.848308e+07  5.848308e+07  1.000000  5.848308e+07  5.848308e+07
3   metric      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31    2021-03-24 14:30:00 CST         both  2.040136e+09  2.040136e+09  1.000000  2.040136e+09  2.040136e+09
4   metric      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31    2021-03-24 14:30:00 CST         both  5.853378e+07  5.853378e+07  1.000000  5.853378e+07  5.853378e+07
5   metric      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31    2021-03-24 14:30:00 CST         both  2.040136e+09  2.040136e+09  1.000000  2.040136e+09  2.040136e+09
6   metric      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31    2021-03-24 14:30:00 CST         both  3.000000e-01  3.000000e-01  1.000000  3.000000e-01  3.000000e-01
7   metric      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31    2021-03-24 14:30:00 CST         both  2.040136e+09  2.040136e+09  1.000000  2.040136e+09  2.040136e+09
8   metric      MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31    2021-03-24 14:30:00 CST         both  3.000000e-01  3.000000e-01  1.000000  3.000000e-01  3.000000e-01
9   metric   Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31    2021-03-24 14:30:00 CST         both  4.000000e-01  3.000000e-01  1.000000  4.000000e-01  3.000000e-01
10  metric   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31    2021-03-24 14:30:00 CST         both  4.000000e+00  4.000000e+00  1.000000  4.000000e+00  4.000000e+00
11  metric  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31    2021-03-24 14:30:00 CST         both  8.230000e+02  8.230000e+02  1.000000  8.230000e+02  8.230000e+02
12  metric  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31    2021-03-24 14:30:00 CST         both  3.978080e+05  3.978080e+05  1.000000  3.978080e+05  3.978080e+05
13  metric  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31    2021-03-24 14:30:00 CST         both  5.000000e+02  5.000000e+02  1.000000  5.000000e+02  5.000000e+02
14  metric  Tomcat02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31    2021-03-24 14:30:00 CST         both  1.000000e+00  1.000000e+00  1.000000  1.000000e+00  1.000000e+00
15  metric  Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31    2021-03-24 14:30:00 CST         both  4.000000e-01  3.000000e-01  1.000000  4.000000e-01  3.000000e-01
16  metric  Tomcat02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31    2021-03-24 14:30:00 CST         both  3.000000e+00  3.000000e+00  1.000000  3.000000e+00  3.000000e+00
17  metric  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31    2021-03-24 14:30:00 CST         both  7.730000e+02  7.730000e+02  1.000000  7.730000e+02  7.730000e+02
18  metric  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31    2021-03-24 14:30:00 CST         both  3.906190e+05  3.906190e+05  1.000000  3.906190e+05  3.906190e+05
19  metric  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31    2021-03-24 14:30:00 CST         both  5.000000e+02  5.000000e+02  1.000000  5.000000e+02  5.000000e+02, (1904, 13))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_df, combined_agg_full, anomalies_full, thresholds (if present)
# Ensure thresholds exists; if not, reconstruct p5 from previous merged raw_all if available.
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

# Use thresholds variable if it exists (it was computed earlier). Otherwise derive p95/p5 by merging.
if 'thresholds' not in globals():
    # attempt to reconstruct p5 as before using metric_df full series
    # Note: This should not recalc thresholds AFTER filtering; it's global from full series.
    raw_metric_full = metric_df.copy()
    # ensure dt tz
    raw_metric_full['dt'] = pd.to_datetime(raw_metric_full['dt']).dt.tz_convert(tz)
    # compute p5 per (source='metric', cmdb_id, kpi)
    p5_m = raw_metric_full.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index().rename(columns={'kpi_name':'kpi','value':'p5'})
    # get p95/p50 from combined_agg_full for metric source
    thresh_m = combined_agg_full[combined_agg_full['source']=='metric'][['source','cmdb_id','kpi','p95','p50']].copy()
    thresholds = thresh_m.merge(p5_m, on=['cmdb_id','kpi'], how='left')
else:
    # ensure thresholds has p95 and p5; keep only metric source
    thresholds = thresholds[thresholds['source']=='metric'][['source','cmdb_id','kpi','p95','p5']].copy()

# Select top 10 metric series by anomaly_count from anomalies_full
top_metric_series = anomalies_full[anomalies_full['source']=='metric'].sort_values('anomaly_count', ascending=False).head(10)
top_keys = top_metric_series[['cmdb_id','kpi']].drop_duplicates()

# Prepare container for all segments
segments = []

# Ensure metric_df dt is timezone-aware Asia/Shanghai
metric_df['dt'] = pd.to_datetime(metric_df['dt']).dt.tz_convert(tz)

# Process each top series
for _, row in top_keys.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi']
    # filter raw metric rows for this series and within time window
    df = metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==kpi)].copy()
    if df.empty:
        continue
    df = df[(df['dt'] >= start_dt) & (df['dt'] <= end_dt)].sort_values('dt').reset_index(drop=True)
    if df.empty:
        continue
    # attach thresholds p95 and p5 for this series from thresholds table
    th = thresholds[(thresholds['cmdb_id']==cmdb) & (thresholds['kpi']==kpi)]
    if th.shape[0]==0:
        # no thresholds available, skip
        continue
    p95 = float(th['p95'].iloc[0]) if pd.notna(th['p95'].iloc[0]) else np.nan
    p5  = float(th['p5'].iloc[0])  if 'p5' in th.columns and pd.notna(th['p5'].iloc[0]) else np.nan

    # label anomalies per row
    def label_val(v):
        high = (not np.isnan(p95)) and (v >= p95)
        low  = (not np.isnan(p5))  and (v <= p5)
        if high and low:
            return 'both'
        if high:
            return 'high'
        if low:
            return 'low'
        return None

    df['anomaly_type'] = df['value'].apply(label_val)
    anom = df[df['anomaly_type'].notna()].copy()
    if anom.empty:
        continue

    # group consecutive-minute anomaly rows into segments (diff == 60s)
    anom = anom.sort_values('dt').reset_index(drop=True)
    anom['ts'] = anom['dt'].astype('int64') // 10**9  # epoch seconds
    anom['prev_ts'] = anom['ts'].shift(1)
    anom['gap'] = anom['ts'] - anom['prev_ts']
    # start a new segment when gap != 60 or first row
    anom['new_seg'] = (anom['gap'] != 60) | (anom['gap'].isna())
    anom['seg_id'] = anom['new_seg'].cumsum()

    # aggregate per segment
    grp = anom.groupby('seg_id').agg(
        cmdb_id = ('cmdb_id','first'),
        kpi = ('kpi_name','first'),
        start_ts = ('dt','first'),
        end_ts = ('dt','last'),
        length_in_minutes = ('dt','count'),
        anomaly_types = ('anomaly_type', lambda s: sorted(set(s))),
        max_value = ('value','max'),
        min_value = ('value','min')
    ).reset_index(drop=True)

    # compute severity and breach ratios per segment using global p95/p5
    def compute_severity_max_breach(r):
        max_v = r['max_value']
        min_v = r['min_value']
        # breach ratios
        breach_h = (max_v - p95)/p95 if (not np.isnan(p95) and p95!=0) else np.nan
        breach_l = (p5 - min_v)/p5 if (not np.isnan(p5) and p5!=0) else np.nan
        # severity definitions: high severity = max_v / p95; low severity = p5 / min_v
        sev_h = max_v / p95 if (not np.isnan(p95) and p95!=0) else np.nan
        sev_l = p5 / min_v if (not np.isnan(p5) and min_v!=0 and not np.isnan(p5)) else np.nan
        max_breach = np.nanmax([breach_h if not np.isnan(breach_h) else -np.inf,
                                breach_l if not np.isnan(breach_l) else -np.inf])
        # determine anomaly_type
        tset = set(r['anomaly_types'])
        if tset == {'high'}:
            atype = 'high'
        elif tset == {'low'}:
            atype = 'low'
        else:
            atype = 'both'
        # severity value per instructions: for high segments severity = max_value/p95; for low severity = p5/min_value; for mixed use max of both
        if atype == 'high':
            severity = sev_h
        elif atype == 'low':
            severity = sev_l
        else:
            severity = np.nanmax([sev_h if not np.isnan(sev_h) else -np.inf,
                                  sev_l if not np.isnan(sev_l) else -np.inf])
            if severity == -np.inf:
                severity = np.nan
        return pd.Series({
            'anomaly_type': atype,
            'breach_h': breach_h,
            'breach_l': breach_l,
            'max_breach': (max_breach if max_breach != -np.inf else np.nan),
            'severity': severity
        })

    sev_df = grp.apply(compute_severity_max_breach, axis=1)
    segs = pd.concat([grp, sev_df], axis=1)
    # attach p95 and p5 and format timestamps
    segs['p95'] = p95
    segs['p5'] = p5
    segs['start_timestamp'] = segs['start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    segs['end_timestamp']   = segs['end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    segs = segs.rename(columns={'max_value':'max_value','min_value':'min_value'})
    # filter out segments whose maximum breach ratio <= 0.5
    # (keep only where max_breach > 0.5)
    segs_filtered = segs[segs['max_breach'] > 0.5].copy()
    if segs_filtered.empty:
        continue
    # select and order columns as requested
    segs_filtered = segs_filtered[['cmdb_id','kpi','start_timestamp','end_timestamp','length_in_minutes',
                                   'anomaly_type','max_value','min_value','severity','p95','p5']].copy()
    # round numeric columns
    for c in ['max_value','min_value','severity','p95','p5']:
        if c in segs_filtered.columns:
            segs_filtered[c] = segs_filtered[c].round(6)
    # append to segments list
    segments.append(segs_filtered)

# Combine all segments into faults_metric_full
if segments:
    faults_metric_full = pd.concat(segments, ignore_index=True)
else:
    faults_metric_full = pd.DataFrame(columns=['cmdb_id','kpi','start_timestamp','end_timestamp','length_in_minutes',
                                               'anomaly_type','max_value','min_value','severity','p95','p5'])

# Sort and prepare top20 by length_in_minutes desc then severity desc
faults_top20 = faults_metric_full.sort_values(['length_in_minutes','severity'], ascending=[False,False]).head(20).reset_index(drop=True)

# Compact display: show top20 segments and the full count
faults_top20, faults_metric_full.shape

```
Out[3]:
```
# Summarize metric-level fault identification results as a single plain-English string.
# Reuse existing kernel variable: faults_metric_full (created in previous step)
tz_note = "UTC+8"
time_window = "2021-03-24 14:30 to 2021-03-24 15:00 (UTC+8)"

# Determine how many metric series were processed (best-effort)
if 'top_keys' in globals():
    num_series = len(top_keys)
elif 'top_metric_series' in globals():
    num_series = top_metric_series.shape[0]
else:
    # fallback: infer from anomalies_full
    num_series = int(anomalies_full[anomalies_full['source']=='metric'].shape[0] if 'anomalies_full' in globals() else 0)

segments_count = int(faults_metric_full.shape[0]) if 'faults_metric_full' in globals() else 0

summary = (
    f"Metric-level fault identification summary for window {time_window}:\n"
    f"- Top {num_series} metric series were analyzed.\n"
    f"- After labeling point anomalies, grouping consecutive-minute anomaly rows into fault segments,\n"
    f"  and filtering segments whose maximum breach ratio <= 0.5, no fault segments remained.\n"
    f"- Stored full segments table in variable 'faults_metric_full' (rows = {segments_count}).\n"
    f"- Conclusion: no metric fault segments meet the breach severity threshold (>50% breach) in the given window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi, start_timestamp, end_timestamp, length_in_minutes, anomaly_type, max_value, min_value, severity, p95, p5]
Index: [], (0, 11))```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing kernel variable anomalies_full
# Filter for source 'trace' or 'log'
anomalies_trace_log = anomalies_full[anomalies_full['source'].isin(['trace','log'])].copy()

# Ensure required columns exist and select/rename as requested (kpi already present)
cols = ['source','cmdb_id','kpi','anomaly_count','earliest_anomaly_timestamp',
        'anomaly_type','max_value','min_value','severity','p95','p5']
# Keep only existing columns (defensive)
cols = [c for c in cols if c in anomalies_trace_log.columns]
anomalies_trace_log = anomalies_trace_log[cols].reset_index(drop=True)

# Sort by anomaly_count desc then severity desc (put NaNs last)
anomalies_trace_log = anomalies_trace_log.sort_values(by=['anomaly_count','severity'], ascending=[False,False], na_position='last').reset_index(drop=True)

# Compact top 20 view
anomalies_trace_log_head = anomalies_trace_log.head(20)

# (1) counts of anomalies per source (trace/log) as a small DataFrame
anomaly_counts_per_source = anomalies_trace_log['source'].value_counts().rename_axis('source').reset_index(name='series_count')

# (2) unique components present in these trace/log anomalies as a small DataFrame
unique_components = sorted(anomalies_trace_log['cmdb_id'].dropna().unique())
unique_components_df = pd.DataFrame({'component': unique_components})

# Return the compact table and the two auxiliary summaries
anomalies_trace_log_head, anomaly_counts_per_source, unique_components_df

```
Out[4]:
```
```python
# Summarize trace/log anomalies in plain English using existing kernel variables:
# anomalies_trace_log_head, anomaly_counts_per_source, unique_components_df, anomalies_trace_log

total = int(anomalies_trace_log.shape[0])
counts = dict(zip(anomaly_counts_per_source['source'], anomaly_counts_per_source['series_count']))
components = unique_components_df['component'].tolist()
earliest = anomalies_trace_log['earliest_anomaly_timestamp'].min()
latest = anomalies_trace_log['earliest_anomaly_timestamp'].max()

# Prepare top example lines (up to 5)
ex_rows = anomalies_trace_log_head.head(5)[['source','cmdb_id','kpi','anomaly_count','anomaly_type','severity']].fillna('NaN')
examples = []
for r in ex_rows.itertuples(index=False):
    examples.append(f"{r.source}|{r.cmdb_id}|{r.kpi}|count={int(r.anomaly_count)}|type={r.anomaly_type}|sev={r.severity}")

summary = (
    f"Trace/Log anomaly summary for window 2021-03-24 14:30 to 2021-03-24 15:00 (UTC+8):\n"
    f"- Total anomalous series (trace+log): {total}.\n"
    f"- By source: trace={counts.get('trace',0)}, log={counts.get('log',0)}.\n"
    f"- Components involved ({len(components)}): {', '.join(components)}.\n"
    f"- Time span of reported anomalies: earliest={earliest}, latest={latest} (UTC+8).\n"
    f"- Top example anomalous series (source|cmdb_id|kpi|count|type|severity):\n  - " + "\n  - ".join(examples) + "\n"
    f"- Note: p95 and p5 thresholds from the precomputed aggregate were used to label anomalies."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   source   cmdb_id                             kpi  anomaly_count earliest_anomaly_timestamp anomaly_type  max_value  min_value    severity         p95          p5
0     log   Mysql01                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
1     log  Tomcat01                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
2     log  Tomcat02                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
3     log  Tomcat03                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
4     log  Tomcat04                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
5     log  apache01                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
6     log  apache02                 log.error_count             31    2021-03-24 14:30:00 CST         both     0.0000    0.00000         NaN    0.000000    0.000000
7   trace  Tomcat03         trace.self.duration_p95             14    2021-03-24 14:30:00 CST         both     1.0040    0.02425    1.706186    1.004000    0.041375
8     log  Tomcat03                log.gc_pause_max              9    2021-03-24 14:34:00 CST         both     0.2500    0.05000    2.500000    0.100000    0.060000
9     log  Tomcat01                   log.row_count              5    2021-03-24 14:38:00 CST         both   557.0000    7.00000   60.857143  553.000000  426.000000
10  trace      MG02   trace.from_dockerB2.row_count              5    2021-03-24 14:35:00 CST         both   168.0000    2.00000   22.500000  135.000000   45.000000
11  trace      IG02     trace.to_Tomcat03.row_count              5    2021-03-24 14:30:00 CST         both   267.0000    6.00000   12.666667  263.500000   76.000000
12  trace  Tomcat03       trace.from_IG02.row_count              5    2021-03-24 14:30:00 CST         both   267.0000    6.00000   12.666667  263.500000   76.000000
13    log  Tomcat02                log.gc_pause_max              5    2021-03-24 14:45:00 CST         both     0.2000    0.03000    1.538462    0.130000    0.034000
14    log  Tomcat04                log.gc_pause_max              5    2021-03-24 14:43:00 CST         both     0.2100    0.04000    1.082474    0.194000    0.040000
15  trace      MG02  trace.to_dockerB2.duration_p95              4    2021-03-24 14:33:00 CST         both     7.2308    0.00490  264.183673    5.240625    1.294500
16  trace      MG02     trace.to_dockerA2.row_count              4    2021-03-24 14:32:00 CST         both   468.0000    1.00000   97.000000  411.500000   97.000000
17  trace      MG01   trace.from_Tomcat01.row_count              4    2021-03-24 14:32:00 CST         both   281.0000    2.00000   90.000000  275.500000  180.000000
18  trace  Tomcat01         trace.to_MG01.row_count              4    2021-03-24 14:32:00 CST         both   281.0000    2.00000   90.000000  275.500000  180.000000
19  trace      IG02     trace.to_Tomcat01.row_count              4    2021-03-24 14:32:00 CST         both   289.0000    2.00000   89.000000  274.500000  178.000000,   source  series_count
0  trace           186
1    log            18,    component
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
5   Tomcat01
6   Tomcat02
7   Tomcat03
8   Tomcat04
9   apache01
10  apache02)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: combined_agg_full, anomalies_trace_log, trace_df (loaded earlier)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

# Ensure trace_df dt is timezone-aware Asia/Shanghai
trace_df['dt'] = pd.to_datetime(trace_df['dt']).dt.tz_convert(tz)

# Prepare thresholds for trace series from combined_agg_full
trace_thresholds = combined_agg_full[combined_agg_full['source']=='trace'][['source','cmdb_id','kpi','p95','p50']].copy()
trace_thresholds = trace_thresholds.rename(columns={'kpi':'trace_name'})

# Compute global p5 for trace series using full trace_df (allowed: global thresholds computed from full series)
p5_full = trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'p5'})
# Merge p5 into trace_thresholds
trace_thresholds = trace_thresholds.merge(p5_full, on=['cmdb_id','trace_name'], how='left')

# Select up to top 10 trace rows from anomalies_trace_log by anomaly_count desc
top_trace_candidates = anomalies_trace_log[anomalies_trace_log['source']=='trace'].sort_values('anomaly_count', ascending=False).head(10)
top_trace_keys = top_trace_candidates[['cmdb_id','kpi']].drop_duplicates().rename(columns={'kpi':'trace_name'})

# Container for segments
all_segments = []

# Iterate each selected trace series
for _, r in top_trace_keys.iterrows():
    cmdb = r['cmdb_id']
    tname = r['trace_name']
    # Get thresholds for this series
    th = trace_thresholds[(trace_thresholds['cmdb_id']==cmdb) & (trace_thresholds['trace_name']==tname)]
    if th.empty:
        continue
    p95 = float(th['p95'].iloc[0]) if pd.notna(th['p95'].iloc[0]) else np.nan
    p5  = float(th['p5'].iloc[0])  if 'p5' in th.columns and pd.notna(th['p5'].iloc[0]) else np.nan

    # Load raw trace rows for this cmdb_id and trace_name within the window
    df = trace_df[(trace_df['cmdb_id']==cmdb) & (trace_df['trace_name']==tname)].copy()
    if df.empty:
        continue
    df = df[(df['dt'] >= start_dt) & (df['dt'] <= end_dt)].sort_values('dt').reset_index(drop=True)
    if df.empty:
        continue

    # Label anomalies using global thresholds
    def label_val(v):
        high = (not np.isnan(p95)) and (v >= p95)
        low  = (not np.isnan(p5))  and (v <= p5)
        if high and low:
            return 'both'
        if high:
            return 'high'
        if low:
            return 'low'
        return None

    df['anomaly_type'] = df['value'].apply(label_val)
    an = df[df['anomaly_type'].notna()].copy()
    if an.empty:
        continue

    # Group consecutive anomalous rows into segments where successive timestamps differ by exactly 60 seconds
    an = an.sort_values('dt').reset_index(drop=True)
    an['ts'] = an['dt'].astype('int64') // 10**9
    an['prev_ts'] = an['ts'].shift(1)
    an['gap'] = an['ts'] - an['prev_ts']
    an['new_seg'] = (an['gap'] != 60) | (an['gap'].isna())
    an['seg_id'] = an['new_seg'].cumsum()

    # Aggregate per segment
    grp = an.groupby('seg_id').agg(
        cmdb_id = ('cmdb_id','first'),
        trace_name = ('trace_name','first'),
        start_ts = ('dt','first'),
        end_ts = ('dt','last'),
        length_in_minutes = ('dt','count'),
        anomaly_types = ('anomaly_type', lambda s: sorted(set(s))),
        max_value = ('value','max'),
        min_value = ('value','min')
    ).reset_index(drop=True)

    # Compute severity per segment
    def seg_severity(row):
        max_v = row['max_value']
        min_v = row['min_value']
        # severity definitions
        sev_h = max_v / p95 if (not np.isnan(p95) and p95!=0) else np.nan
        sev_l = p5 / min_v if (not np.isnan(p5) and min_v!=0 and not np.isnan(p5)) else np.nan
        tset = set(row['anomaly_types'])
        if tset == {'high'}:
            atype = 'high'
            severity = sev_h
        elif tset == {'low'}:
            atype = 'low'
            severity = sev_l
        else:
            atype = 'both'
            severity = np.nanmax([sev_h if not np.isnan(sev_h) else -np.inf,
                                  sev_l if not np.isnan(sev_l) else -np.inf])
            if severity == -np.inf:
                severity = np.nan
        return pd.Series({'anomaly_type': atype, 'severity': severity})

    sev = grp.apply(seg_severity, axis=1)
    segs = pd.concat([grp, sev], axis=1)

    # Attach p95 and p5, format timestamps
    segs['p95'] = p95
    segs['p5'] = p5
    segs['start_timestamp'] = segs['start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    segs['end_timestamp']   = segs['end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Filter out segments whose severity <= 1.5 (keep only severity > 1.5)
    segs_filtered = segs[segs['severity'] > 1.5].copy()
    if segs_filtered.empty:
        continue

    # Select requested columns and round numerics
    segs_filtered = segs_filtered[['cmdb_id','trace_name','start_timestamp','end_timestamp',
                                   'length_in_minutes','anomaly_type','max_value','min_value','severity','p95','p5']].copy()
    for c in ['max_value','min_value','severity','p95','p5']:
        if c in segs_filtered.columns:
            segs_filtered[c] = segs_filtered[c].round(6)

    all_segments.append(segs_filtered)

# Combine into full table and store in kernel variable
if all_segments:
    traces_faults_full = pd.concat(all_segments, ignore_index=True)
else:
    traces_faults_full = pd.DataFrame(columns=['cmdb_id','trace_name','start_timestamp','end_timestamp',
                                               'length_in_minutes','anomaly_type','max_value','min_value','severity','p95','p5'])

# Prepare compact result: top 20 rows sorted by length desc then severity desc
traces_top20 = traces_faults_full.sort_values(['length_in_minutes','severity'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display compact result and the full shape
traces_top20, traces_faults_full.shape

```
Out[5]:
```
```python
# Summarize trace-level fault identification results in plain English as a single string.
# Reuse existing kernel variables: traces_faults_full (full table) and traces_top20 (compact view)

# Prepare summary details
total_segments = int(traces_faults_full.shape[0]) if 'traces_faults_full' in globals() else 0
time_window = "2021-03-24 14:30 to 2021-03-24 15:00 (UTC+8)"

# If there are segments, extract top examples
examples = []
if total_segments > 0:
    top = traces_top20.head(5) if 'traces_top20' in globals() else traces_faults_full.head(5)
    for r in top.itertuples(index=False):
        examples.append(f"{r.cmdb_id}|{r.trace_name}|{r.start_timestamp}→{r.end_timestamp}|len={int(r.length_in_minutes)}m|type={r.anomaly_type}|sev={r.severity}")
else:
    examples = ["none"]

summary = (
    f"Trace-level fault identification summary for window {time_window}:\n"
    f"- Total fault segments detected after severity filtering (>1.5): {total_segments}.\n"
    f"- Most notable segment: Tomcat03 trace.self.duration_p95 had a 3-minute segment from 2021-03-24 14:58 to 15:00 (UTC+8) labeled 'both' with severity ≈ 1.706.\n"
    f"- Other segments are single-minute low-severity breaches at 15:00 on components: MG02, MG01, IG02, Tomcat01, etc. (examples below).\n"
    f"- Top example segments (cmdb_id|trace_name|start→end|len|type|severity):\n  - " + "\n  - ".join(examples) + "\n"
    f"- Full segments table is stored in variable 'traces_faults_full'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                      trace_name          start_timestamp            end_timestamp  length_in_minutes anomaly_type  max_value  min_value    severity         p95          p5
0  Tomcat03         trace.self.duration_p95  2021-03-24 14:58:00 CST  2021-03-24 15:00:00 CST                  3         both     1.0040    0.02425    1.706186    1.004000    0.041375
1      MG02  trace.to_dockerB2.duration_p95  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     0.0049    0.00490  264.183673    5.240625    1.294500
2      MG02     trace.to_dockerA2.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     1.0000    1.00000   97.000000  411.500000   97.000000
3      MG01   trace.from_Tomcat01.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     2.0000    2.00000   90.000000  275.500000  180.000000
4  Tomcat01         trace.to_MG01.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     2.0000    2.00000   90.000000  275.500000  180.000000
5      IG02     trace.to_Tomcat01.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     2.0000    2.00000   89.000000  274.500000  178.000000
6  Tomcat01       trace.from_IG02.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     2.0000    2.00000   89.000000  274.500000  178.000000
7      MG02   trace.from_dockerB2.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     2.0000    2.00000   22.500000  135.000000   45.000000
8      IG02     trace.to_Tomcat03.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     6.0000    6.00000   12.666667  263.500000   76.000000
9  Tomcat03       trace.from_IG02.row_count  2021-03-24 15:00:00 CST  2021-03-24 15:00:00 CST                  1          low     6.0000    6.00000   12.666667  263.500000   76.000000, (10, 11))```
```

In [ ]:
# In[6]:

import pytz
import pandas as pd

# reuse trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

# ensure dt is timezone-aware Asia/Shanghai
trace_df['dt'] = pd.to_datetime(trace_df['dt']).dt.tz_convert(tz)

# filter rows for Tomcat03 and the specific trace within the time window
tomcat03_trace_self = trace_df[
    (trace_df['cmdb_id'] == 'Tomcat03') &
    (trace_df['trace_name'] == 'trace.self.duration_p95') &
    (trace_df['dt'] >= start_dt) &
    (trace_df['dt'] <= end_dt)
].loc[:, ['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').reset_index(drop=True)

# show up to 50 rows (stored in kernel variable tomcat03_trace_self)
tomcat03_trace_self.head(50)

```
Out[6]:
```
```python
# Summarize the extracted Tomcat03 trace rows in plain English as a single string.
import pytz
import pandas as pd

# Reuse existing kernel variable: tomcat03_trace_self, anomalies_trace_log
tz = pytz.timezone('Asia/Shanghai')

df = tomcat03_trace_self.copy()

# convert timestamps to timezone-aware datetimes (Asia/Shanghai)
df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# basic stats
row_count = int(df.shape[0])
start_ts = df['dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
end_ts   = df['dt'].max().strftime('%Y-%m-%d %H:%M:%S %Z')
min_val = float(df['value'].min())
max_val = float(df['value'].max())

# fetch p95 and p5 from anomalies_trace_log if available
p95 = None
p5 = None
if 'anomalies_trace_log' in globals():
    match = anomalies_trace_log[(anomalies_trace_log['source']=='trace') &
                                (anomalies_trace_log['cmdb_id']=='Tomcat03') &
                                (anomalies_trace_log['kpi']=='trace.self.duration_p95')]
    if not match.empty:
        p95 = float(match['p95'].iloc[0]) if 'p95' in match.columns and pd.notna(match['p95'].iloc[0]) else None
        p5  = float(match['p5'].iloc[0])  if 'p5' in match.columns  and pd.notna(match['p5'].iloc[0])  else None

# count anomalies relative to thresholds if thresholds exist
high_count = low_count = 0
earliest_high = earliest_low = None
if p95 is not None or p5 is not None:
    highs = df[df['value'] >= p95] if p95 is not None else df.iloc[0:0]
    lows  = df[df['value'] <= p5]  if p5  is not None else df.iloc[0:0]
    high_count = int(highs.shape[0])
    low_count  = int(lows.shape[0])
    if high_count>0:
        earliest_high = highs['dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
    if low_count>0:
        earliest_low = lows['dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z')

summary = (
    f"Tomcat03 trace.self.duration_p95 rows (window {start_ts} to {end_ts}):\n"
    f"- Total rows: {row_count}.\n"
    f"- Time span: {start_ts} → {end_ts} (Asia/Shanghai, UTC+8).\n"
    f"- Value range: min={min_val}, max={max_val}.\n"
)
if p95 is not None and p5 is not None:
    summary += (
        f"- Thresholds used: p95={p95}, p5={p5}.\n"
        f"- High anomalies (value >= p95): {high_count}" + (f", earliest at {earliest_high}." if earliest_high else ".") + "\n"
        f"- Low anomalies (value <= p5): {low_count}"  + (f", earliest at {earliest_low}."  if earliest_low  else ".") + "\n"
    )
else:
    summary += "- Thresholds (p95/p5) not found in anomalies_trace_log; anomaly counts not computed.\n"

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id               trace_name    value
0   1616567400  Tomcat03  trace.self.duration_p95  1.00400
1   1616567460  Tomcat03  trace.self.duration_p95  0.60215
2   1616567520  Tomcat03  trace.self.duration_p95  0.44000
3   1616567580  Tomcat03  trace.self.duration_p95  1.00400
4   1616567640  Tomcat03  trace.self.duration_p95  1.00300
..         ...       ...                      ...      ...
26  1616568960  Tomcat03  trace.self.duration_p95  0.70555
27  1616569020  Tomcat03  trace.self.duration_p95  0.71500
28  1616569080  Tomcat03  trace.self.duration_p95  1.00400
29  1616569140  Tomcat03  trace.self.duration_p95  1.00400
30  1616569200  Tomcat03  trace.self.duration_p95  0.02425

[31 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Time window in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

# Ensure log_df exists (was loaded earlier). If not, load from CSV.
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure error_logs is loaded
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')

# Ensure dt column is timezone-aware Asia/Shanghai for log_df
if 'dt' in log_df.columns:
    log_df['dt'] = pd.to_datetime(log_df['dt']).dt.tz_convert(tz)
else:
    log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter Tomcat03 logs in the window
tomcat03_logs = log_df[
    (log_df['cmdb_id'] == 'Tomcat03') &
    (log_df['dt'] >= start_dt) &
    (log_df['dt'] <= end_dt)
].loc[:, ['timestamp','cmdb_id','log_name','value']].sort_values('timestamp').reset_index(drop=True)

# Prepare error logs: ensure dt and timezone
if 'dt' in error_logs_df.columns:
    error_logs_df['dt'] = pd.to_datetime(error_logs_df['dt']).dt.tz_convert(tz)
else:
    error_logs_df['dt'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter Tomcat03 error logs in the window
tomcat03_error_logs = error_logs_df[
    (error_logs_df['cmdb_id'] == 'Tomcat03') &
    (error_logs_df['dt'] >= start_dt) &
    (error_logs_df['dt'] <= end_dt)
].loc[:, ['timestamp','cmdb_id','message']].sort_values('timestamp').reset_index(drop=True)

# Display compact results (limit 50 rows) and keep full filtered DataFrames stored in kernel variables
tomcat03_logs.head(50), tomcat03_error_logs.head(50)

```
Out[7]:
```
```python
# Summarize Tomcat03 logs and error messages in plain English using existing kernel variables:
# tomcat03_logs, tomcat03_error_logs

import pytz
import pandas as pd

tz = pytz.timezone('Asia/Shanghai')

df = tomcat03_logs.copy()
err = tomcat03_error_logs.copy()

# ensure datetime column
df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

total_rows = int(df.shape[0])
earliest = df['dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z') if total_rows>0 else None
latest   = df['dt'].max().strftime('%Y-%m-%d %H:%M:%S %Z') if total_rows>0 else None
log_names = sorted(df['log_name'].unique().tolist())

# error_count summary
ec = df[df['log_name']=='log.error_count']
ec_nonzero_count = int((ec['value'] > 0).sum())
ec_max = float(ec['value'].max()) if not ec.empty else None

# gc_pause_max summary
gc = df[df['log_name']=='log.gc_pause_max']
gc_max = float(gc['value'].max()) if not gc.empty else None
gc_max_time = gc.loc[gc['value'].idxmax(),'dt'].strftime('%Y-%m-%d %H:%M:%S %Z') if (not gc.empty) else None
gc_examples = gc[['dt','value']].sort_values('dt').head(3).to_dict('records') if not gc.empty else []

# row_count summary
rc = df[df['log_name']=='log.row_count']
rc_min = float(rc['value'].min()) if not rc.empty else None
rc_max = float(rc['value'].max()) if not rc.empty else None
rc_mean = float(rc['value'].mean()) if not rc.empty else None

# error logs textual messages
err_count = int(err.shape[0])
err_preview = err.head(3).to_dict('records') if err_count>0 else []

summary = (
    f"Tomcat03 logs 2021-03-24 14:30–15:00 (UTC+8):\n"
    f"- Total log rows: {total_rows}. Time span: {earliest} → {latest}.\n"
    f"- Log types present: {', '.join(log_names)}.\n"
    f"- log.error_count: non-zero occurrences = {ec_nonzero_count}, max={ec_max}.\n"
    f"- log.gc_pause_max: max={gc_max} at {gc_max_time}. Example samples (earliest up to 3): {gc_examples}.\n"
    f"- log.row_count: min={rc_min}, max={rc_max}, mean≈{round(rc_mean,3) if rc_mean is not None else None}.\n"
    f"- Error messages: total entries = {err_count}."
)
if err_count>0:
    summary += " Preview: " + str(err_preview)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id          log_name   value
0   1616567400  Tomcat03   log.error_count    0.00
1   1616567400  Tomcat03  log.gc_pause_max    0.07
2   1616567400  Tomcat03     log.row_count  476.00
3   1616567460  Tomcat03   log.error_count    0.00
4   1616567460  Tomcat03     log.row_count  505.00
..         ...       ...               ...     ...
45  1616568360  Tomcat03  log.gc_pause_max    0.25
46  1616568360  Tomcat03     log.row_count  523.00
47  1616568420  Tomcat03   log.error_count    0.00
48  1616568420  Tomcat03  log.gc_pause_max    0.07
49  1616568420  Tomcat03     log.row_count  488.00

[50 rows x 4 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# time window (inclusive) in UTC+8
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:56:00"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00:00"))

# reuse cached tomcat03_logs if present, otherwise load log.csv
try:
    tomcat03_logs
except NameError:
    log_df = pd.read_csv('log.csv')
    log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    tomcat03_logs = log_df[(log_df['cmdb_id']=='Tomcat03')].loc[:, ['timestamp','cmdb_id','log_name','value']].sort_values('timestamp').reset_index(drop=True)

# create timezone-aware datetime column for filtering
tomcat03_logs['dt'] = pd.to_datetime(tomcat03_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# filter rows in the fault window
tomcat03_logs_fault_window = tomcat03_logs[(tomcat03_logs['dt'] >= start_dt) & (tomcat03_logs['dt'] <= end_dt)].loc[:, ['timestamp','cmdb_id','log_name','value']].sort_values('timestamp').reset_index(drop=True)

# Determine global p95 for (source=log, cmdb_id=Tomcat03, kpi=log.gc_pause_max) from combined_agg_full if available
p95_row = combined_agg_full[
    (combined_agg_full['source']=='log') &
    (combined_agg_full['cmdb_id']=='Tomcat03') &
    (combined_agg_full['kpi']=='log.gc_pause_max')
]

if not p95_row.empty and pd.notna(p95_row['p95'].iloc[0]):
    p95_val = float(p95_row['p95'].iloc[0])
else:
    # fallback: compute global p95 from full log.csv series for Tomcat03 log.gc_pause_max (global, not window-limited)
    try:
        log_df
    except NameError:
        log_df = pd.read_csv('log.csv')
    full_series = log_df[(log_df['cmdb_id']=='Tomcat03') & (log_df['log_name']=='log.gc_pause_max')]['value'].astype(float)
    p95_val = float(full_series.quantile(0.95)) if not full_series.empty else np.nan

# Check gc_pause_max rows in the window
gc_rows = tomcat03_logs_fault_window[tomcat03_logs_fault_window['log_name']=='log.gc_pause_max'].copy()
gc_breach_count = 0
earliest_breach_ts = None
max_gc_in_window = float(gc_rows['value'].max()) if not gc_rows.empty else np.nan

if (not np.isnan(p95_val)) and (not gc_rows.empty):
    breaches = gc_rows[gc_rows['value'] >= p95_val].copy()
    gc_breach_count = int(breaches.shape[0])
    if gc_breach_count > 0:
        # earliest breach timestamp in Asia/Shanghai formatted
        earliest_breach_ts = pd.to_datetime(breaches['timestamp'], unit='s', utc=True).dt.tz_convert(tz).min().strftime('%Y-%m-%d %H:%M:%S %Z')

# Build a compact one-line aggregate summary as a single-row DataFrame
gc_breach_summary = pd.DataFrame([{
    'breach_exists': (gc_breach_count > 0),
    'breach_count': gc_breach_count,
    'earliest_breach_timestamp': earliest_breach_ts,
    'max_gc_pause_in_window': round(max_gc_in_window,6) if not np.isnan(max_gc_in_window) else np.nan,
    'p95_global': round(p95_val,6) if not np.isnan(p95_val) else np.nan
}])

# Return the filtered logs table (limit 50 rows) and the compact breach summary
tomcat03_logs_fault_window.head(50), gc_breach_summary

```
Out[8]:
```
```python
# Summarize findings for Tomcat03 in the 14:56–15:00 (UTC+8) window as a single plain-English string.
# Reuse cached variables: tomcat03_logs_fault_window, gc_breach_summary, tomcat03_error_logs

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# basic stats from the filtered logs table
logs = tomcat03_logs_fault_window.copy()
row_count = int(logs.shape[0])
if row_count > 0:
    start_dt = pd.to_datetime(logs['timestamp'].iloc[0], unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    end_dt   = pd.to_datetime(logs['timestamp'].iloc[-1], unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    start_dt = end_dt = None

# gc breach summary
gb = gc_breach_summary.iloc[0]
breach_exists = bool(gb['breach_exists'])
breach_count = int(gb['breach_count'])
earliest_breach = gb['earliest_breach_timestamp']
max_gc = gb['max_gc_pause_in_window']
p95_val = gb['p95_global']

# error logs presence
err_count = int(tomcat03_error_logs.shape[0]) if 'tomcat03_error_logs' in globals() else 0

# notable row_count low value in the window (if present)
rc = logs[logs['log_name']=='log.row_count']
min_row_count = int(rc['value'].min()) if not rc.empty else None
min_row_ts = (pd.to_datetime(rc.loc[rc['value'].idxmin(),'timestamp'], unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
              if (not rc.empty) else None)

summary = (
    f"Tomcat03 logs in 2021-03-24 14:56–15:00 (UTC+8):\n"
    f"- Total log rows in window: {row_count}. Time span shown: {start_dt} → {end_dt}.\n"
    f"- GC pauses: global p95 for log.gc_pause_max = {p95_val}. Breaches present: {breach_exists}.\n"
    f"  - Breach count: {breach_count}. Earliest breach: {earliest_breach}. Max gc_pause_max in window: {max_gc}.\n"
    f"- Row counts: min row_count in window = {min_row_count}" + (f" at {min_row_ts}." if min_row_ts else ".") + "\n"
    f"- Error log messages: total entries = {err_count} (none found in the error_logs dataset for this window).\n"
    f"Conclusion: GC pause reached the global p95 (0.1s) twice (earliest at {earliest_breach}); also a sharp row_count drop to {min_row_count} observed at the end of the window. No textual error messages were recorded in error_logs for Tomcat03 in this window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id          log_name   value
0   1616568960  Tomcat03   log.error_count    0.00
1   1616568960  Tomcat03  log.gc_pause_max    0.10
2   1616568960  Tomcat03     log.row_count  521.00
3   1616569020  Tomcat03   log.error_count    0.00
4   1616569020  Tomcat03  log.gc_pause_max    0.10
5   1616569020  Tomcat03     log.row_count  490.00
6   1616569080  Tomcat03   log.error_count    0.00
7   1616569080  Tomcat03     log.row_count  540.00
8   1616569140  Tomcat03   log.error_count    0.00
9   1616569140  Tomcat03  log.gc_pause_max    0.08
10  1616569140  Tomcat03     log.row_count  511.00
11  1616569200  Tomcat03   log.error_count    0.00
12  1616569200  Tomcat03     log.row_count    8.00,    breach_exists  breach_count earliest_breach_timestamp  max_gc_pause_in_window  p95_global
0           True             2   2021-03-24 14:56:00 CST                     0.1         0.1)```
```

In [ ]:
# In[9]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[9]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# ensure metric_df is loaded (reuse if present)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# timezone (UTC+8) for any datetime handling (not strictly needed for this aggregation)
tz = pytz.timezone('Asia/Shanghai')

# filter for Tomcat03 (full file, no time filtering)
tomcat03_metrics = metric_df[metric_df['cmdb_id'] == 'Tomcat03'].copy()

# ensure numeric values
tomcat03_metrics['value'] = pd.to_numeric(tomcat03_metrics['value'], errors='coerce')
tomcat03_metrics = tomcat03_metrics.dropna(subset=['value', 'kpi_name'])

# group by kpi_name and aggregate
g = tomcat03_metrics.groupby('kpi_name')['value']
agg_basic = g.agg(sample_count='count', min='min', max='max', mean='mean')
quantiles = g.quantile([0.5, 0.9, 0.95, 0.99]).unstack(level=-1).rename(columns={0.5:'p50', 0.9:'p90', 0.95:'p95', 0.99:'p99'})
tomcat03_metric_kpis = agg_basic.join(quantiles).reset_index().rename(columns={'kpi_name':'kpi_name'})

# round numeric columns for compactness
num_cols = ['min','max','mean','p50','p90','p95','p99']
for c in num_cols:
    if c in tomcat03_metric_kpis.columns:
        tomcat03_metric_kpis[c] = tomcat03_metric_kpis[c].round(6)

# sort and limit to top 50
tomcat03_metric_kpis = tomcat03_metric_kpis.sort_values(['sample_count','kpi_name'], ascending=[False,True]).reset_index(drop=True)
tomcat03_metric_kpis_head = tomcat03_metric_kpis.head(50)

# store full aggregated table in required kernel variable
# (tomcat03_metric_kpis already defined as the full table)
tomcat03_metric_kpis_head, tomcat03_metric_kpis.shape

```
Out[10]:
```
```python
# Summarize the Tomcat03 metric KPI aggregation in plain English.
# Reuse existing kernel variables: tomcat03_metric_kpis (full) and tomcat03_metric_kpis_head (compact)

top_table = tomcat03_metric_kpis_head.copy()
total_kpis = int(tomcat03_metric_kpis.shape[0])

top5 = top_table.head(5).loc[:, ['kpi_name','sample_count','min','max','mean','p95']]

summary_lines = [
    f"Tomcat03 metric aggregation (full file): {total_kpis} unique KPI series aggregated.",
    "Top KPIs by sample count (examples):"
]
for r in top5.itertuples(index=False):
    summary_lines.append(f"- {r.kpi_name}: samples={int(r.sample_count)}, min={r.min}, max={r.max}, mean≈{r.mean}, p95={r.p95}")

summary_lines.append("Note: many core CPU-related KPIs (CPULoad, CPUSysTime, CPUUserTime, CPUWio, CPUidleutil) have full coverage (31 samples).")
summary_lines.append("Filesystem-related KPIs appear with fewer samples (e.g., 15 samples for some FS metrics).")
summary_lines.append("Full aggregated table stored in 'tomcat03_metric_kpis' for further inspection.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  sample_count        min        max       mean        p50        p90        p95        p99
0                             OSLinux-CPU_CPU_CPULoad            31   1.010000   1.320000   1.102903   1.100000   1.170000   1.200000   1.284000
1                          OSLinux-CPU_CPU_CPUSysTime            31   0.657320   1.706201   0.803383   0.786026   0.881900   0.921750   1.473971
2                         OSLinux-CPU_CPU_CPUUserTime            31  25.894800  26.545984  26.071327  26.058000  26.202964  26.220500  26.448399
3                              OSLinux-CPU_CPU_CPUWio            31   0.029129   1.618800   0.510268   0.345400   1.094420   1.374584   1.566170
4                         OSLinux-CPU_CPU_CPUidleutil            31  70.349563  72.841869  72.131607  72.312600  72.676959  72.749590  72.820880
..                                                ...           ...        ...        ...        ...        ...        ...        ...        ...
45     OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity            15  15.416955  15.416955  15.416955  15.416955  15.416955  15.416955  15.416955
46  OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUse...            15   0.471497   0.471497   0.471497   0.471497   0.471497   0.471497   0.471497
47  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCa...            15   0.426812   0.426812   0.426812   0.426812   0.426812   0.426812   0.426812
48  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...            15   0.004196   0.004196   0.004196   0.004196   0.004196   0.004196   0.004196
49     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity            15   3.337943   3.338973   3.338635   3.338890   3.338957   3.338973   3.338973

[50 rows x 9 columns], (121, 9))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables: metric_df, tomcat03_metric_kpis (aggregated full KPI list)
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is loaded
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure dt is timezone-aware Asia/Shanghai
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Use the cached full KPI aggregation for Tomcat03
try:
    tomcat03_metric_kpis
except NameError:
    raise RuntimeError("tomcat03_metric_kpis not found in kernel. Run previous aggregation step.")

# Identify KPI names matching substrings (case-insensitive)
substrs = ['JVM','GC','Tomcat','Threads','CPU','Memory']
pattern = '|'.join(substrs)
matched_kpis = tomcat03_metric_kpis[tomcat03_metric_kpis['kpi_name'].str.contains(pattern, case=False, regex=True)]['kpi_name'].unique().tolist()

# Prepare global p95 (from tomcat03_metric_kpis) and compute global p5 using full series (rule 9)
p95_df = tomcat03_metric_kpis[['kpi_name','p95']].rename(columns={'kpi_name':'kpi_name','p95':'p95_global'})
p5_series = metric_df[metric_df['cmdb_id']=='Tomcat03'].groupby('kpi_name')['value'].quantile(0.05).reset_index().rename(columns={'value':'p5_global'})

thresholds = p95_df.merge(p5_series, left_on='kpi_name', right_on='kpi_name', how='left')

# Filter raw rows for Tomcat03, selected KPIs, and the given time window
start_dt = tz.localize(pd.to_datetime("2021-03-24 14:30"))
end_dt   = tz.localize(pd.to_datetime("2021-03-24 15:00"))

tomcat03_metric_window_rows = metric_df[
    (metric_df['cmdb_id']=='Tomcat03') &
    (metric_df['kpi_name'].isin(matched_kpis)) &
    (metric_df['dt'] >= start_dt) &
    (metric_df['dt'] <= end_dt)
].copy().sort_values(['kpi_name','dt']).reset_index(drop=True)

# Aggregate per kpi_name in the window
def summarize_kpi(group):
    kpi = group.name
    scount = int(group.shape[0])
    min_v = float(group['value'].min()) if scount>0 else np.nan
    max_v = float(group['value'].max()) if scount>0 else np.nan
    mean_v = float(group['value'].mean()) if scount>0 else np.nan
    earliest = group['dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z') if scount>0 else None
    latest = group['dt'].max().strftime('%Y-%m-%d %H:%M:%S %Z') if scount>0 else None
    # global thresholds
    th = thresholds[thresholds['kpi_name']==kpi]
    p95 = float(th['p95_global'].iloc[0]) if (not th.empty and pd.notna(th['p95_global'].iloc[0])) else np.nan
    p5  = float(th['p5_global'].iloc[0])  if (not th.empty and 'p5_global' in th.columns and pd.notna(th['p5_global'].iloc[0])) else np.nan
    # breaches
    highs = group[group['value'] >= p95] if (not np.isnan(p95)) else group.iloc[0:0]
    lows  = group[group['value'] <= p5]  if (not np.isnan(p5)) else group.iloc[0:0]
    breach_count = int(highs.shape[0] + lows.shape[0])
    earliest_breach_ts = None
    if breach_count>0:
        # find earliest between highs and lows
        times = pd.to_datetime(pd.concat([highs['dt'], lows['dt']])).sort_values()
        earliest_breach_ts = times.iloc[0].strftime('%Y-%m-%d %H:%M:%S %Z')
    # severities
    max_sev_high = float((highs['value'] / p95).max()) if (not highs.empty and p95!=0 and not np.isnan(p95)) else np.nan
    max_sev_low  = float((p5 / lows['value']).max()) if (not lows.empty and p5!=0 and not np.isnan(p5)) else np.nan
    # choose max severity (considering both)
    candidates = [x for x in [max_sev_high, max_sev_low] if not pd.isna(x)]
    max_breach_severity = float(np.nan) if len(candidates)==0 else float(np.max(candidates))
    return pd.Series({
        'kpi_name': kpi,
        'sample_count_in_window': scount,
        'min_value': round(min_v,6) if not np.isnan(min_v) else np.nan,
        'max_value': round(max_v,6) if not np.isnan(max_v) else np.nan,
        'mean_value': round(mean_v,6) if not np.isnan(mean_v) else np.nan,
        'earliest_timestamp_in_window': earliest,
        'latest_timestamp_in_window': latest,
        'global_p95': round(p95,6) if not np.isnan(p95) else np.nan,
        'global_p5': round(p5,6) if not np.isnan(p5) else np.nan,
        'breach_count': breach_count,
        'earliest_breach_timestamp': earliest_breach_ts,
        'max_breach_severity': round(max_breach_severity,6) if not pd.isna(max_breach_severity) else np.nan
    })

if not tomcat03_metric_window_rows.empty:
    tomcat03_metric_window_summary = tomcat03_metric_window_rows.groupby('kpi_name').apply(summarize_kpi).reset_index(drop=True)
else:
    # empty summary with columns
    cols = ['kpi_name','sample_count_in_window','min_value','max_value','mean_value',
            'earliest_timestamp_in_window','latest_timestamp_in_window','global_p95','global_p5',
            'breach_count','earliest_breach_timestamp','max_breach_severity']
    tomcat03_metric_window_summary = pd.DataFrame(columns=cols)

# Sort by max_breach_severity desc and limit top 20
tomcat03_metric_window_summary = tomcat03_metric_window_summary.sort_values(['max_breach_severity','sample_count_in_window'], ascending=[False,False]).reset_index(drop=True)
tomcat03_metric_window_summary_head = tomcat03_metric_window_summary.head(20)

# Ensure full per-row extracted data saved in kernel variable as requested
# (tomcat03_metric_window_rows already contains full rows)

# Display compact outputs
tomcat03_metric_window_summary_head, tomcat03_metric_window_rows.shape

```
Out[11]:
```
```python
# Summarize the Tomcat03 JVM/CPU/Threads/Memory/GC KPI window analysis in plain English.
# Reuse existing kernel variables: tomcat03_metric_window_summary, tomcat03_metric_window_rows

import pandas as pd

# defensive access
summary_df = tomcat03_metric_window_summary if 'tomcat03_metric_window_summary' in globals() else pd.DataFrame()
rows_df = tomcat03_metric_window_rows if 'tomcat03_metric_window_rows' in globals() else pd.DataFrame()

num_kpis = int(summary_df.shape[0])
total_rows = int(rows_df.shape[0])

# Top 5 KPIs by max_breach_severity
if not summary_df.empty:
    top5 = summary_df.sort_values('max_breach_severity', ascending=False).head(5)[
        ['kpi_name','sample_count_in_window','breach_count','max_breach_severity','earliest_breach_timestamp']
    ].fillna('N/A')
    top_lines = []
    for r in top5.itertuples(index=False):
        top_lines.append(f"{r.kpi_name} | samples={int(r.sample_count_in_window)} | breaches={int(r.breach_count)} | max_sev={r.max_breach_severity} | earliest_breach={r.earliest_breach_timestamp}")
else:
    top_lines = ["none"]

summary = (
    f"Tomcat03 metric window summary (2021-03-24 14:30–15:00 UTC+8):\n"
    f"- KPI series considered matching JVM/GC/Tomcat/Threads/CPU/Memory: {num_kpis}.\n"
    f"- Total raw metric rows extracted for these KPIs in window: {total_rows}.\n"
    f"- Top KPIs by max breach severity (kpi | samples | breaches | max_sev | earliest_breach):\n  - " + "\n  - ".join(top_lines) + "\n"
    f"- Notes: Highest-severity KPI is '{top5.iloc[0].kpi_name}' (if present). Severity >1 indicates values exceeding global p95 (high) or falling below p5 (low).\n"
    f"- Full per-row data stored in 'tomcat03_metric_window_rows' and the KPI summary table stored in 'tomcat03_metric_window_summary'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  sample_count_in_window     min_value     max_value    mean_value earliest_timestamp_in_window latest_timestamp_in_window    global_p95     global_p5  breach_count earliest_breach_timestamp  max_breach_severity
0                     OSLinux-CPU_CPU-1_SingleCpuidle                      15  0.000000e+00  9.733910e+01  9.013155e+01      2021-03-24 14:30:00 CST    2021-03-24 14:59:00 CST  9.730418e+01  6.623355e+01             2   2021-03-24 14:38:00 CST                  inf
1                          OSLinux-CPU_CPU_CPUSysTime                      31  6.573200e-01  1.706201e+00  8.033830e-01      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  9.217500e-01  6.761620e-01             4   2021-03-24 14:37:00 CST             1.851045
2                              OSLinux-CPU_CPU_CPUWio                      31  2.912900e-02  1.618800e+00  5.102680e-01      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  1.374584e+00  4.368100e-02             4   2021-03-24 14:33:00 CST             1.499588
3   Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31  0.000000e+00  1.300000e+01  4.129032e+00      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  9.000000e+00  0.000000e+00             6   2021-03-24 14:30:00 CST             1.444444
4                     OSLinux-CPU_CPU-2_SingleCpuUtil                       8  2.945100e+00  1.000000e+02  6.389894e+01      2021-03-24 14:33:00 CST    2021-03-24 14:57:00 CST  1.000000e+02  3.255305e+00             5   2021-03-24 14:33:00 CST             1.105329
5                             OSLinux-CPU_CPU_CPULoad                      31  1.010000e+00  1.320000e+00  1.102903e+00      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  1.200000e+00  1.020000e+00             7   2021-03-24 14:32:00 CST             1.100000
6                     OSLinux-CPU_CPU-1_SingleCpuUtil                       9  2.828100e+00  5.380600e+00  3.683444e+00      2021-03-24 14:33:00 CST    2021-03-24 14:57:00 CST  5.081080e+00  2.889380e+00             2   2021-03-24 14:51:00 CST             1.058948
7                     OSLinux-CPU_CPU-0_SingleCpuUtil                       9  2.795300e+00  1.000000e+02  4.646659e+01      2021-03-24 14:33:00 CST    2021-03-24 14:57:00 CST  1.000000e+02  2.948620e+00             4   2021-03-24 14:35:00 CST             1.054849
8                     OSLinux-CPU_CPU-3_SingleCpuUtil                       8  3.129200e+00  5.816700e+00  4.054550e+00      2021-03-24 14:33:00 CST    2021-03-24 14:57:00 CST  5.547760e+00  3.221985e+00             2   2021-03-24 14:41:00 CST             1.048477
9             Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                       6  1.357235e+09  2.063560e+09  1.658459e+09      2021-03-24 14:31:00 CST    2021-03-24 14:56:00 CST  1.989872e+09  1.380903e+09             2   2021-03-24 14:51:00 CST             1.037032
10     Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent                       5  3.200000e+01  4.900000e+01  3.860000e+01      2021-03-24 14:34:00 CST    2021-03-24 14:59:00 CST  4.760000e+01  3.240000e+01             2   2021-03-24 14:54:00 CST             1.029412
11            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                       5  2.124059e+09  2.830385e+09  2.551230e+09      2021-03-24 14:34:00 CST    2021-03-24 14:59:00 CST  2.811450e+09  2.183627e+09             2   2021-03-24 14:54:00 CST             1.028044
12           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem                      18  1.180000e+02  1.360000e+02  1.266111e+02      2021-03-24 14:31:00 CST    2021-03-24 15:00:00 CST  1.334500e+02  1.197000e+02             2   2021-03-24 14:38:00 CST             1.019108
13                         OSLinux-CPU_CPU_CPUCpuUtil                      17  2.722810e+01  2.965040e+01  2.792880e+01      2021-03-24 14:32:00 CST    2021-03-24 15:00:00 CST  2.910776e+01  2.730402e+01             2   2021-03-24 14:38:00 CST             1.018642
14                        OSLinux-CPU_CPU_CPUUserTime                      31  2.589480e+01  2.654598e+01  2.607133e+01      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  2.622050e+01  2.590677e+01             4   2021-03-24 14:37:00 CST             1.012413
15                        OSLinux-CPU_CPU_CPUidleutil                      31  7.034956e+01  7.284187e+01  7.213161e+01      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  7.274959e+01  7.118426e+01             4   2021-03-24 14:33:00 CST             1.011865
16  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31  3.670000e+02  3.880000e+02  3.800645e+02      2021-03-24 14:30:00 CST    2021-03-24 15:00:00 CST  3.865000e+02  3.695000e+02             4   2021-03-24 14:30:00 CST             1.006812
17                    OSLinux-CPU_CPU-3_SingleCpuidle                      15  9.418333e+01  9.687084e+01  9.611454e+01      2021-03-24 14:30:00 CST    2021-03-24 14:59:00 CST  9.686864e+01  9.472118e+01             2   2021-03-24 14:40:00 CST             1.005711
18                    OSLinux-CPU_CPU-0_SingleCpuidle                      15  0.000000e+00  9.720466e+01  6.397387e+01      2021-03-24 14:30:00 CST    2021-03-24 14:59:00 CST  9.693640e+01  0.000000e+00             5   2021-03-24 14:34:00 CST             1.002767
19                    OSLinux-CPU_CPU-2_SingleCpuidle                      15  0.000000e+00  9.705490e+01  3.853917e+01      2021-03-24 14:30:00 CST    2021-03-24 14:59:00 CST  9.689110e+01  0.000000e+00             9   2021-03-24 14:30:00 CST             1.001691, (1075, 5))```
```